In [272]:
import pandas as pd
import matplotlib as plt
import numpy as np
import seaborn as sns
import os
from sklearn.model_selection import train_test_split


In [273]:
base_data_path= "data"
print(f"data location {os.listdir(base_data_path)}")
complete_datapath = os.path.join(base_data_path, "train.csv")

print(f"complete data path {complete_datapath}")

data location ['test.csv', 'train.csv', 'validation.csv']
complete data path data\train.csv


In [274]:
df= pd.read_csv(complete_datapath)
# print(df.head())

df['IsChurn'] = df['Churn'].map({0: 'No', 1: 'Yes'})
df.to_csv('data_Pro/processed_data.csv', index=False)

In [275]:
assert (df['Monthly Charge'] >= 0).all(), "Monthly Charge contains negative values!"

In [276]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary>0]
print(missing_summary)

Churn Reason                         3104
Churn Category                       3104
Offer                                2324
Internet Type                         886
Avg Monthly Long Distance Charges       0
Avg Monthly GB Download                 0
Churn Score                             0
CLTV                                    0
City                                    0
Country                                 0
Customer ID                             0
Customer Status                         0
Contract                                0
Dependents                              0
Device Protection Plan                  0
Gender                                  0
Internet Service                        0
Lat Long                                0
Latitude                                0
Longitude                               0
Age                                     0
Married                                 0
Monthly Charge                          0
Number of Dependents              

In [277]:
num_cols =  df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,4225.0,46.451124,16.731518,19.000000,32.000000,46.000000,60.000000,80.000000
Avg Monthly GB Download,4225.0,20.740828,20.366105,0.000000,4.000000,17.000000,27.000000,85.000000
Avg Monthly Long Distance Charges,4225.0,22.766963,15.429992,0.000000,9.050000,22.570000,36.170000,49.990000
Churn Score,4225.0,58.281183,21.197931,5.000000,40.000000,61.000000,75.000000,96.000000
CLTV,4225.0,4409.751243,1170.599119,2003.000000,3493.000000,4531.000000,5381.000000,6500.000000
Dependents,4225.0,0.233136,0.422878,0.000000,0.000000,0.000000,0.000000,1.000000
Device Protection Plan,4225.0,0.346982,0.476066,0.000000,0.000000,0.000000,1.000000,1.000000
Internet Service,4225.0,0.790296,0.407146,0.000000,1.000000,1.000000,1.000000,1.000000
Latitude,4225.0,36.207274,2.471090,32.555828,33.994524,36.205465,38.196497,41.962127
Longitude,4225.0,-119.768187,2.154078,-124.301372,-121.788090,-119.622676,-117.991372,-114.192901


In [278]:

x = df.drop(columns=['Churn', 'Customer ID'])
y = df['Churn']

x_train , X_test, y_Train, y_test = train_test_split(x, y, 
                                                     test_size=0.2, 
                                                     random_state=42, 
                                                     stratify=y)

print(f"X_train shape: {x_train.shape}, y_train shape: {y_Train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (3380, 51), y_train shape: (3380,)
X_test shape: (845, 51), y_test shape: (845,)


In [279]:
df['Churn'].value_counts(normalize=True)


Churn
0    0.734675
1    0.265325
Name: proportion, dtype: float64

In [280]:
q1 = df['Monthly Charge'].quantile(0.25)
q3 = df['Monthly Charge'].quantile(0.75)
iqr = q3 - q1
df[(df['Monthly Charge'] < (q1 - 1.5 * iqr)) | (df['Monthly Charge'] > (q3 + 1.5 * iqr))]

,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,Churn Category,Churn Reason,Churn Score,City,CLTV,Contract,Country,...,Total Charges,Total Extra Data Charges,Total Long Distance Charges,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code,Churn,IsChurn


In [281]:
df.groupby("Contract").agg(
    customers=("Customer ID", "count"),
    churn_rate=("Churn", "mean"),
    avg_revenue=("Total Revenue", "mean"),
    avg_tenure=("Tenure in Months", "mean")
)

,customers,churn_rate,avg_revenue,avg_tenure
Contract,,,,
Month-to-Month,2193,0.454628,1735.921281,17.688554
One Year,904,0.107301,4098.387179,42.180310
Two Year,1128,0.023936,4823.781906,54.218972


In [282]:
df["state_churn_rate"] = df.groupby("State")["Churn Score"].rank(ascending=False)
df.sort_values(["State", "state_churn_rate"]).head(20)

,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,Churn Category,Churn Reason,Churn Score,City,CLTV,Contract,Country,...,Total Extra Data Charges,Total Long Distance Charges,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code,Churn,IsChurn,state_churn_rate
497,66,23,38.78,Competitor,Competitor had better devices,96,Moss Landing,4843,Month-to-Month,United States,...,0,38.78,0.00,124.83,0,1,95039,1,Yes,14.0
807,72,17,8.92,Attitude,Attitude of service provider,96,Nuevo,4969,Month-to-Month,United States,...,0,62.44,32.46,557.88,0,1,92567,1,Yes,14.0
813,45,26,6.40,Attitude,Attitude of support person,96,Lancaster,2181,Month-to-Month,United States,...,10,38.40,0.00,511.45,0,0,93536,1,Yes,14.0
868,72,15,34.51,Competitor,Competitor had better devices,96,Hermosa Beach,3438,Month-to-Month,United States,...,0,34.51,0.00,104.11,0,1,90254,1,Yes,14.0
914,33,28,3.51,Competitor,Competitor made better offer,96,San Diego,4415,Month-to-Month,United States,...,100,203.58,0.00,6188.98,0,0,92117,1,Yes,14.0
936,37,13,24.63,Other,Don't know,96,Keene,3579,Month-to-Month,United States,...,0,369.45,0.00,1639.00,0,1,93531,1,Yes,14.0
1358,74,24,11.31,Competitor,Competitor offered more data,96,Greenbrae,4777,Month-to-Month,United States,...,0,158.34,0.00,1424.44,0,1,94904,1,Yes,14.0
1491,63,20,43.32,Competitor,Competitor had better devices,96,Armona,3223,Month-to-Month,United States,...,10,173.28,0.00,504.93,0,0,93202,1,Yes,14.0
2026,74,4,39.36,Competitor,Competitor made better offer,96,Cutler,5362,Month-to-Month,United States,...,0,1220.16,0.00,4406.81,0,1,93615,1,Yes,14.0
2076,39,30,18.99,Attitude,Attitude of service provider,96,Douglas City,5915,Month-to-Month,United States,...,0,303.84,0.00,1859.49,0,1,96024,1,Yes,14.0


In [283]:
df["TenureBucket"]= pd.cut(
    df["Tenure in Months"],
    bins=[0, 12, 24, 36, 48, 60, np.inf],
    labels=["0-12", "13-24", "25-36", "37-48", "49-60", "60+"]
)
df.groupby("TenureBucket")["Churn"].mean()

TenureBucket
0-12     0.475825
13-24    0.283806
25-36    0.229645
37-48    0.202479
49-60    0.148297
60+      0.056911
Name: Churn, dtype: float64

In [284]:
demographics = df[["Customer ID","Age","Gender","Senior Citizen","Married"]]
pilling = df[["Customer ID","Monthly Charge", "Total Charges", "Total Revenue"]]

merged = pd.merge(demographics, pilling, on="Customer ID", how="inner")
merged.head()

,Customer ID,Age,Gender,Senior Citizen,Married,Monthly Charge,Total Charges,Total Revenue
0,4526-ZJJTM,72,Female,1,1,88.40,2191.15,2677.15
1,5302-BDJNT,27,Male,0,0,95.50,3418.20,5014.90
2,5468-BPMMO,59,Male,0,1,19.60,851.20,1590.42
3,2212-LYASK,25,Male,0,1,45.85,1246.40,1276.40
4,0378-XSZPU,31,Male,0,1,60.30,3563.80,4562.56


In [285]:
pd.pivot_table(
    df,
    index="State", 
    values="Total Revenue",
    columns="Contract", 
    aggfunc="mean"
)

Contract,Month-to-Month,One Year,Two Year
State,,,
California,1735.921281,4098.387179,4823.781906


In [286]:
df["MonthlyCharge_scaled"] = (df["Monthly Charge"] - df['Monthly Charge'].mean())
print(f"Scaled monthly charge: {df['MonthlyCharge_scaled'].head()}")


Scaled monthly charge: 0    23.493538
1    30.593538
2   -45.306462
3   -19.056462
4    -4.606462
Name: MonthlyCharge_scaled, dtype: float64


In [ ]:
model_df.to_parquet('data_Pro/model_data.parquet', index=False) 